# 🇹🇲 FEMSeg Туркменский v1.1 — чтение из папок с чанками

**Обновлено:** ноутбук читает готовые сегментированные папки

**Доступные корпуса на Drive (`SEGMENTATION_TURKM/`):**
| Папка | Предложений |
|-------|-------------|
| `turkmen_80000_segmented` | ~80 000 |
| `turkmen_50000_segmented` | ~50 000 |
| `turkmen_40000_segmented` | ~40 000 |
| `turkmen_10000_segmented` | ~10 000 |

**Формат чанков** (`chunk_000.txt`, `chunk_001.txt`, ...):
```
adam@@ lar@@ yň geldi
öý@@ ler@@ iň
Türkmenistan@@ da ýerleşýän
```

**Порядок:** Ячейки 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10

## 📁 Ячейка 1 — Google Drive

In [ ]:
from google.colab import drive
import os

# Монтируем Drive только если ещё не смонтирован
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('✅ Google Drive уже смонтирован')

SEG_DIR    = '/content/drive/MyDrive/SEGMENTATION_TURKM'
MODEL_BASE = '/content/drive/MyDrive/TURKM_MORPH'
os.makedirs(MODEL_BASE, exist_ok=True)
print(f'✅ Корпуса: {SEG_DIR}')
print(f'✅ Модели:  {MODEL_BASE}')


## 📦 Ячейка 2 — Установка зависимостей

In [ ]:
!pip install -q pytorch-crf
print('✅ pytorch-crf установлен')


## 🗂️ Ячейка 2b — Объединение корпусов в 200k (запустить один раз)

Копирует чанки из всех четырёх корпусов в одну папку `turkmen_200000_segmented`.
**Запускайте только один раз** — повторный запуск пропустит уже существующую папку.

In [ ]:
import os, glob, shutil

SEG_DIR = '/content/drive/MyDrive/SEGMENTATION_TURKM'

# Исходные папки — порядок важен для нумерации чанков
SRC_FOLDERS = [
    'turkmen_80000_segmented',
    'turkmen_50000_segmented',
    'turkmen_40000_segmented',
    'turkmen_10000_segmented',
]

DST_NAME = 'turkmen_200000_segmented'
DST_DIR  = os.path.join(SEG_DIR, DST_NAME)

# Проверяем — если папка уже есть и не пустая, пропускаем
existing = sorted(glob.glob(os.path.join(DST_DIR, 'chunk_*.txt')))
if existing:
    print(f'✅ Папка уже существует: {DST_DIR}')
    print(f'   Чанков: {len(existing)}')
    # Считаем строки
    total = sum(sum(1 for _ in open(f, encoding='utf-8')) for f in existing)
    print(f'   Строк:  {total:,}')
    print(f'\n   Пропускаем создание — папка готова к использованию.')
else:
    os.makedirs(DST_DIR, exist_ok=True)
    print(f'📂 Создаём: {DST_DIR}')
    print()

    chunk_n = 0
    total_lines = 0

    for folder_name in SRC_FOLDERS:
        src_path = os.path.join(SEG_DIR, folder_name)
        if not os.path.exists(src_path):
            print(f'  ⚠️  Не найдена: {folder_name} — пропускаем')
            continue

        chunks = sorted(glob.glob(os.path.join(src_path, 'chunk_*.txt')))
        if not chunks:
            chunks = sorted(glob.glob(os.path.join(src_path, '*.txt')))

        folder_lines = 0
        for src_file in chunks:
            dst_file = os.path.join(DST_DIR, f'chunk_{chunk_n:04d}.txt')
            shutil.copy2(src_file, dst_file)
            with open(dst_file, encoding='utf-8') as f:
                n = sum(1 for _ in f)
            folder_lines += n
            total_lines  += n
            chunk_n += 1

        print(f'  ✅ {folder_name:<35} {len(chunks):>3} чанков  {folder_lines:>8,} строк')

    print()
    print(f'✅ Готово!')
    print(f'   Папка:   {DST_DIR}')
    print(f'   Чанков:  {chunk_n}')
    print(f'   Строк:   {total_lines:,}')
    print(f'\n   Теперь в Ячейке 3 установите:')
    print(f'   CORPUS_NAME = \'turkmen_200000_segmented\'')


## ✏️ Ячейка 3 — Выбор корпуса

**Единственное место где нужно что-то менять вручную.**

Укажите название папки в строке `CORPUS_NAME = '...'`

Все остальные пути рассчитываются автоматически.

In [ ]:
import os, glob, json

# =============================================================
# ШАГ 1: найдём реальный путь к MyDrive
# =============================================================
MYDRIVE = '/content/drive/MyDrive'
if not os.path.exists(MYDRIVE):
    raise FileNotFoundError(
        'Google Drive не смонтирован.\n'
        'Запустите Ячейку 1 сначала.'
    )

# Показываем содержимое MyDrive чтобы найти нужную папку
print('📂 Содержимое MyDrive:')
for d in sorted(os.listdir(MYDRIVE)):
    full = os.path.join(MYDRIVE, d)
    mark = '📁' if os.path.isdir(full) else '📄'
    print(f'  {mark} {d}')

# =============================================================
# ШАГ 2: укажите точное название папки с корпусами
# Посмотрите список выше и скопируйте точное имя папки
# =============================================================
SEG_FOLDER = 'SEGMENTATION_TURKM'   # <-- исправьте если имя другое

# =============================================================
# ШАГ 3: укажите название корпуса внутри папки
# =============================================================
CORPUS_NAME = 'turkmen_200000_segmented'  # <-- меняйте здесь
# Варианты (смотрите что есть в папке):
# 'turkmen_200000_segmented'  ← объединённый (рекомендуется для ablation)
# 'turkmen_80000_segmented'
# 'turkmen_50000_segmented'
# 'turkmen_40000_segmented'
# 'turkmen_10000_segmented'
# =============================================================

SEG_DIR    = os.path.join(MYDRIVE, SEG_FOLDER)
MODEL_BASE = os.path.join(MYDRIVE, 'TURKM_MORPH')
os.makedirs(MODEL_BASE, exist_ok=True)

if not os.path.exists(SEG_DIR):
    print(f'\n❌ Папка не найдена: {SEG_DIR}')
    print(f'   Исправьте SEG_FOLDER выше')
    print(f'   Доступные папки в MyDrive:')
    for d in sorted(os.listdir(MYDRIVE)):
        if os.path.isdir(os.path.join(MYDRIVE, d)):
            print(f'     {d}')
else:
    print(f'\n✅ SEG_DIR найден: {SEG_DIR}')
    print(f'   Содержимое:')
    for d in sorted(os.listdir(SEG_DIR)):
        full = os.path.join(SEG_DIR, d)
        mark = '📁' if os.path.isdir(full) else '📄'
        print(f'     {mark} {d}')

    CORPUS_DIR   = os.path.join(SEG_DIR, CORPUS_NAME)
    safe         = CORPUS_NAME.replace(' ', '_')
    CHAR2ID_PATH = os.path.join(MODEL_BASE, f'char2id_{safe}.json')
    MODEL_DIR    = os.path.join(MODEL_BASE, f'models_{safe}')
    os.makedirs(MODEL_DIR, exist_ok=True)

    # Ищем чанки
    CHUNK_FILES = sorted(glob.glob(os.path.join(CORPUS_DIR, 'chunk_*.txt')))
    if not CHUNK_FILES:
        CHUNK_FILES = sorted(glob.glob(os.path.join(CORPUS_DIR, '*.txt')))

    if not CHUNK_FILES:
        print(f'\n❌ Чанки не найдены в: {CORPUS_DIR}')
        print(f'   Доступные папки в SEG_DIR:')
        for d in sorted(os.listdir(SEG_DIR)):
            if os.path.isdir(os.path.join(SEG_DIR, d)):
                n = len(glob.glob(os.path.join(SEG_DIR, d, '*.txt')))
                print(f'     📁 {d}  ({n} .txt файлов)')
        print(f'\n   Исправьте CORPUS_NAME выше')
    else:
        total_lines = sum(
            sum(1 for _ in open(c, encoding='utf-8'))
            for c in CHUNK_FILES
        )
        print(f'\n✅ Корпус:  {CORPUS_NAME}')
        print(f'✅ Чанков:  {len(CHUNK_FILES)}')
        print(f'✅ Строк:   {total_lines:,}')
        print(f'✅ Модели:  {MODEL_DIR}')
        print(f'\n   Первые чанки:')
        for c in CHUNK_FILES[:4]:
            kb = os.path.getsize(c) // 1024
            print(f'     {os.path.basename(c)}  ({kb} KB)')
        if len(CHUNK_FILES) > 4:
            print(f'     ... ещё {len(CHUNK_FILES)-4}')


## 🎵 Ячейка 4 — Туркменский алфавит и сингармонизм

| Класс | Гласные | Аффиксы |
|-------|---------|----------|
| **Передние** (front) | e, i, ö, ü | -ler, -de, -den, -iň |
| **Задние** (back) | a, y, o, u | -lar, -da, -dan, -yň |


In [ ]:
# Нормализация: ň (U+0148) -> ñ (U+00F1) — КРИТИЧНО
def norm_turkm(s: str) -> str:
    return (str(s)
            .replace('\u0148', '\u00f1')
            .replace('\u0147', '\u00d1')
            .replace('\ufeff', '')
            .strip())

FRONT_VOWELS = set('eiöü')
BACK_VOWELS  = set('ayou')
ALL_VOWELS   = FRONT_VOWELS | BACK_VOWELS
HARMONY_FRONT     = 0
HARMONY_BACK      = 1
HARMONY_CONSONANT = 2

def char_harmony_class(ch):
    c = ch.lower()
    if c in FRONT_VOWELS: return HARMONY_FRONT
    if c in BACK_VOWELS:  return HARMONY_BACK
    return HARMONY_CONSONANT

def word_harmony_class(word):
    for ch in reversed(word.lower()):
        cls = char_harmony_class(ch)
        if cls != HARMONY_CONSONANT: return cls
    return HARMONY_CONSONANT

def is_vowel(ch): return 1 if ch.lower() in ALL_VOWELS else 0

HARMONY_PAIRS = {
    'lar':'ler','ler':'lar','da':'de','de':'da',
    'dan':'den','den':'dan','dy':'di','di':'dy',
    'dyr':'dir','dir':'dyr','yň':'iň','iň':'yň',
    'ny':'ni','ni':'ny','a':'e','e':'a','y':'i','i':'y',
}

def check_harmony(stem, affix):
    stem_h = word_harmony_class(stem)
    if stem_h == HARMONY_CONSONANT: return True
    if affix.lower() not in HARMONY_PAIRS: return True
    if stem_h == HARMONY_FRONT:
        return word_harmony_class(affix) != HARMONY_BACK
    return True

print('🎵 Тест сингармонизма:')
tests = [('kitap','back'),('öý','front'),('adam','back'),
         ('döwlet','front'),('Türkmenistan','back'),('güýç','front')]
for word, expected in tests:
    got = 'front' if word_harmony_class(word)==HARMONY_FRONT else 'back'
    print(f'  {"✅" if got==expected else "❌"} {word:<20} -> {got}')


## 🔤 Ячейка 5 — BMES-теги и разбор строк

In [ ]:
from typing import List, Tuple

TAGS   = ['B', 'M', 'E', 'S']
TAG2ID = {t: i for i, t in enumerate(TAGS)}
ID2TAG = {i: t for t, i in TAG2ID.items()}

def morphs_to_bmes_char(word, morphs):
    tags = []
    for m in morphs:
        n = len(m)
        if n == 1: tags.append('S')
        else: tags += ['B'] + ['M']*(n-2) + ['E']
    return tags if len(tags)==len(word) else []

PUNCT_TOKENS = {',','.','?','!',';',':','—','-','…','«','»','(',')','"'}

def line_to_word_morphs(line):
    tokens = line.strip().split()
    words, cur = [], []
    for tok in tokens:
        if tok in PUNCT_TOKENS:
            if cur:
                morphs = [t[:-2] if t.endswith('@@') else t for t in cur]
                words.append((''.join(morphs), morphs)); cur=[]
            words.append((tok,[tok])); continue
        cur.append(norm_turkm(tok))
        if not tok.endswith('@@'):
            morphs = [t[:-2] if t.endswith('@@') else t for t in cur]
            words.append((''.join(morphs), morphs)); cur=[]
    if cur:
        morphs = [t[:-2] if t.endswith('@@') else t for t in cur]
        words.append((''.join(morphs), morphs))
    return words

print('✅ Функции BMES и разбора строк готовы')


## 📊 Ячейка 6 — Выборка, словарь символов и split

Три шага в одной ячейке:
1. Читает все `chunk_*.txt` из выбранной папки
2. Строит словарь символов `char2id`
3. Делит на train (90%) / val (10%)

In [ ]:
from tqdm import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split

# ── Проверка что корпус выбран ────────────────────────────────────────────
if 'CHUNK_FILES' not in dir() or not CHUNK_FILES:
    raise RuntimeError(
        'CHUNK_FILES не определён.\n'
        'Запустите Ячейку 3 и нажмите кнопку ✅ Выбрать корпус'
    )

# ── Шаг 1: строим выборку из чанков ──────────────────────────────────────
def build_samples_from_chunks(chunk_files: list):
    samples, total_lines, bad_words = [], 0, 0
    print(f'📂 Читаем {len(chunk_files)} чанков из:')
    print(f'   {CORPUS_DIR}')
    for chunk_path in tqdm(chunk_files, desc='Чанки'):
        with open(chunk_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                total_lines += 1
                for word, morphs in line_to_word_morphs(line):
                    chars = list(word)
                    tags  = morphs_to_bmes_char(word, morphs)
                    if not tags: bad_words += 1; continue
                    tag_ids          = [TAG2ID[t] for t in tags]
                    harmony_ids      = [char_harmony_class(ch) for ch in chars]
                    is_vowel_ids     = [is_vowel(ch) for ch in chars]
                    word_h           = word_harmony_class(word)
                    word_harmony_ids = [word_h] * len(chars)
                    samples.append((chars, harmony_ids, is_vowel_ids,
                                     word_harmony_ids, tag_ids))
    print(f'\n  ✅ Строк:        {total_lines:,}')
    print(f'  ✅ Семплов:      {len(samples):,}')
    print(f'  ⚠️  Пропущено:   {bad_words:,}')
    return samples

samples = build_samples_from_chunks(CHUNK_FILES)

# ── Шаг 2: словарь символов ──────────────────────────────────────────────
def build_char_vocab(samples, min_freq=1):
    cnt = Counter()
    for chars, *_ in samples:
        for ch in chars: cnt[ch] += 1
    char2id = {'<pad>': 0, '<unk>': 1}
    for ch, c in cnt.items():
        if c >= min_freq: char2id[ch] = len(char2id)
    special = sorted([ch for ch in char2id if ch in 'çñöşüýžÇÑÖŞÜÝŽ'])
    print(f'\n  Вокабуляр:        {len(char2id)} символов')
    print(f'  Спецбуквы туркм.: {special}')
    return char2id

def encode_samples(samples, char2id):
    enc = []
    for chars, harm, vow, wh, tags in samples:
        ids = [char2id.get(ch, char2id['<unk>']) for ch in chars]
        enc.append((ids, harm, vow, wh, tags))
    return enc

char2id = build_char_vocab(samples)

with open(CHAR2ID_PATH, 'w', encoding='utf-8') as f:
    json.dump(char2id, f, ensure_ascii=False, indent=2)
print(f'  char2id -> {CHAR2ID_PATH}')

# ── Шаг 3: train/val split ───────────────────────────────────────────────
encoded = encode_samples(samples, char2id)
train_data, val_data = train_test_split(encoded, test_size=0.1, random_state=42)
print(f'  Train: {len(train_data):,}  |  Val: {len(val_data):,}')
print(f'\n✅ Данные готовы — можно запускать Ячейку 7 (модель)')


## 🧠 Ячейка 8 — Dataset и модель TurkmFEMSegCRF

```
char_emb    [128d] ┐
harmony_emb  [16d] ├── concat [168d] ──> BiLSTM(2L,256h) ──> CRF
is_vowel_emb  [8d] │
word_harm_emb[16d] ┘
```


In [ ]:
import torch, torch.nn as nn
from torchcrf import CRF
from torch.utils.data import Dataset, DataLoader

class TurkmMorphDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn(batch):
    max_len = max(len(x[0]) for x in batch)
    ac,ah,av,aw,at,am = [],[],[],[],[],[]
    for char_ids, harm, vow, wh, tags in batch:
        l = len(char_ids); p = max_len-l
        ac.append(char_ids+[0]*p)
        ah.append(harm+[HARMONY_CONSONANT]*p)
        av.append(vow+[0]*p)
        aw.append(wh+[HARMONY_CONSONANT]*p)
        at.append(tags+[-1]*p)
        am.append([1]*l+[0]*p)
    return (torch.tensor(ac,dtype=torch.long),
            torch.tensor(ah,dtype=torch.long),
            torch.tensor(av,dtype=torch.long),
            torch.tensor(aw,dtype=torch.long),
            torch.tensor(at,dtype=torch.long),
            torch.tensor(am,dtype=torch.bool))

train_ds = TurkmMorphDataset(train_data)
val_ds   = TurkmMorphDataset(val_data)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, collate_fn=collate_fn)

class TurkmFEMSegCRF(nn.Module):
    def __init__(self, vocab_size, tagset_size,
                 emb_dim=128, harmony_dim=16, vowel_dim=8,
                 word_harm_dim=16, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.char_emb      = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.harmony_emb   = nn.Embedding(3, harmony_dim)
        self.is_vowel_emb  = nn.Embedding(2, vowel_dim)
        self.word_harm_emb = nn.Embedding(3, word_harm_dim)
        in_dim = emb_dim+harmony_dim+vowel_dim+word_harm_dim
        self.lstm    = nn.LSTM(in_dim, hidden_dim, num_layers=2,
                               bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim*2, tagset_size)
        self.crf     = CRF(tagset_size, batch_first=True)

    def forward(self, inp, harm, vow, wh, tags=None, mask=None):
        x = torch.cat([self.char_emb(inp), self.harmony_emb(harm),
                        self.is_vowel_emb(vow), self.word_harm_emb(wh)], dim=-1)
        x, _ = self.lstm(x)
        x     = self.dropout(x)
        emis  = self.fc(x)
        if tags is not None:
            return -self.crf(emis, tags, mask=mask, reduction='mean')
        return self.crf.decode(emis, mask=mask)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = TurkmFEMSegCRF(vocab_size=len(char2id), tagset_size=len(TAGS)).to(device)
n_par  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Device:          {device}')
print(f'Параметров:      {n_par:,}')
print(f'Вход BiLSTM:     {128+16+8+16}d  (128 char + 16 harm + 8 vowel + 16 word_harm)')
print(f'Батчей в трейне: {len(train_loader)}')


## 🚀 Ячейка 9 — Обучение

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2)

def evaluate(model, loader):
    model.eval()
    total_tok, correct_tok = 0, 0
    total_morph, correct_morph = 0, 0
    with torch.no_grad():
        for batch in loader:
            inp,harm,vow,wh,tags,mask = [b.to(device) for b in batch]
            paths = model(inp,harm,vow,wh,tags=None,mask=mask)
            pred  = torch.full_like(tags,-1)
            for i,seq in enumerate(paths):
                for j,t in enumerate(seq): pred[i,j]=t
            valid = mask.view(-1)&(tags.view(-1)>=0)
            total_tok    += valid.sum().item()
            correct_tok  += (tags.view(-1)[valid]==pred.view(-1)[valid]).sum().item()
            for i in range(tags.shape[0]):
                l = mask[i].sum().item()
                total_morph   += 1
                correct_morph += (tags[i,:l].tolist()==pred[i,:l].tolist())
    return correct_tok/max(1,total_tok), correct_morph/max(1,total_morph)

# Автоматически адаптируем кол-во эпох под размер корпуса
n_samples   = len(train_data)
num_epochs  = 3 if n_samples > 500000 else 5 if n_samples > 200000 else 7
corpus_name = os.path.basename(CORPUS_DIR)
best_morph  = 0.0

print(f'── Обучение: {corpus_name} ─────────────────────────────')
print(f'   Train семплов: {len(train_data):,}  |  Val: {len(val_data):,}')
print(f'   Эпох:          {num_epochs}')
print(f'   Device:        {device}')
print()

for epoch in range(1, num_epochs+1):
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch:02d}'):
        inp,harm,vow,wh,tags,mask = [b.to(device) for b in batch]
        loss = model(inp,harm,vow,wh,tags=tags,mask=mask)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
        optimizer.step()
        total_loss += loss.item(); n_batches += 1

    avg_loss = total_loss/max(1,n_batches)
    val_tok, val_morph = evaluate(model, val_loader)
    scheduler.step(val_morph)

    print(f'[Epoch {epoch:02d}]  loss={avg_loss:.4f}  '
          f'token_acc={val_tok:.4f}  morph_acc={val_morph:.4f}')

    ckpt = os.path.join(MODEL_DIR, f'femseg_turkm_{corpus_name}_ep{epoch:02d}.pt')
    torch.save({'epoch':epoch,'state_dict':model.state_dict(),
                'optimizer':optimizer.state_dict(),
                'val_tok':val_tok,'val_morph':val_morph,
                'char2id':char2id,'corpus':corpus_name}, ckpt)
    print(f'  💾 {os.path.basename(ckpt)}')

    if val_morph > best_morph:
        best_morph = val_morph
        best_path  = os.path.join(MODEL_DIR, f'femseg_turkm_{corpus_name}_BEST.pt')
        torch.save({'epoch':epoch,'state_dict':model.state_dict(),
                    'val_tok':val_tok,'val_morph':val_morph,
                    'char2id':char2id,'corpus':corpus_name}, best_path)
        print(f'  ★  BEST morph_acc={best_morph:.4f}')

print(f'\n✅ Готово! Лучший morph_acc = {best_morph:.4f}')
print(f'   Модель: {best_path}')


## 🧪 Ячейка 11 — Ablation Study: вклад признаков сингармонизма

Сравниваем три варианта модели:

| Вариант | Описание | Вход в BiLSTM |
|---------|----------|---------------|
| **A** — базовый | только символьный эмбеддинг | 128d |
| **B** — +char_harmony | + класс гармонии символа | 128+16=144d |
| **C** — полный (наш) | + is_vowel + word_harmony | 128+16+8+16=168d |

**Стратегия:** запускаем на **10k корпусе** (7 эпох) — на малых данных
BiLSTM не успевает выучить гармонию самостоятельно, поэтому вклад
явных harmony-признаков виден максимально чётко.

Затем сравниваем с результатами на 80k для полной картины.

In [ ]:
import torch, torch.nn as nn, os, json, copy
from torchcrf import CRF
from torch.utils.data import DataLoader

# =============================================================
# ВАРИАНТЫ МОДЕЛЕЙ ДЛЯ ABLATION
# =============================================================

class FEMSegVariant(nn.Module):
    """
    Унифицированная модель с управляемым набором признаков.
    use_harmony    — char_harmony_emb (3 класса: front/back/cons)
    use_vowel      — is_vowel_emb (2 класса: гласная/согл.)
    use_word_harm  — word_harmony_emb (3 класса: гармония слова)
    """
    def __init__(self, vocab_size, tagset_size,
                 emb_dim=128, harmony_dim=16, vowel_dim=8,
                 word_harm_dim=16, hidden_dim=256, dropout=0.3,
                 use_harmony=True, use_vowel=True, use_word_harm=True):
        super().__init__()
        self.use_harmony   = use_harmony
        self.use_vowel     = use_vowel
        self.use_word_harm = use_word_harm

        self.char_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        in_dim = emb_dim
        if use_harmony:
            self.harmony_emb = nn.Embedding(3, harmony_dim)
            in_dim += harmony_dim
        if use_vowel:
            self.vowel_emb = nn.Embedding(2, vowel_dim)
            in_dim += vowel_dim
        if use_word_harm:
            self.word_harm_emb = nn.Embedding(3, word_harm_dim)
            in_dim += word_harm_dim

        self.lstm    = nn.LSTM(in_dim, hidden_dim, num_layers=2,
                               bidirectional=True, batch_first=True,
                               dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim*2, tagset_size)
        self.crf     = CRF(tagset_size, batch_first=True)
        self.in_dim  = in_dim

    def forward(self, inp, harm, vow, wh, tags=None, mask=None):
        parts = [self.char_emb(inp)]
        if self.use_harmony:   parts.append(self.harmony_emb(harm))
        if self.use_vowel:     parts.append(self.vowel_emb(vow))
        if self.use_word_harm: parts.append(self.word_harm_emb(wh))
        x = torch.cat(parts, dim=-1)
        x, _ = self.lstm(x)
        x     = self.dropout(x)
        emis  = self.fc(x)
        if tags is not None:
            return -self.crf(emis, tags, mask=mask, reduction='mean')
        return self.crf.decode(emis, mask=mask)


# =============================================================
# ОБУЧЕНИЕ ОДНОГО ВАРИАНТА
# =============================================================

def train_variant(variant_name, model, train_loader, val_loader,
                  num_epochs=7, lr=1e-3, device='cpu'):
    """Обучает вариант и возвращает историю метрик."""
    model = model.to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # Warmup: первые 2 эпохи lr растёт от lr/10 до lr
    def lr_lambda(epoch):
        warmup_epochs = 2
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 1.0
    warmup_sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        opt, mode='max', factor=0.5, patience=2)

    history = []
    best_morph = 0.0
    best_state = None

    print(f'\n── Вариант {variant_name} (вход {model.in_dim}d, lr={lr}) ──────────────────')
    for epoch in range(1, num_epochs+1):
        model.train()
        total_loss, n = 0.0, 0
        for batch in train_loader:
            inp,harm,vow,wh,tags,mask = [b.to(device) for b in batch]
            loss = model(inp,harm,vow,wh,tags=tags,mask=mask)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item(); n += 1

        # Валидация
        model.eval()
        total_tok, correct_tok = 0, 0
        total_morph, correct_morph = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                inp,harm,vow,wh,tags,mask = [b.to(device) for b in batch]
                paths = model(inp,harm,vow,wh,tags=None,mask=mask)
                pred  = torch.full_like(tags,-1)
                for i,seq in enumerate(paths):
                    for j,t in enumerate(seq): pred[i,j]=t
                valid = mask.view(-1)&(tags.view(-1)>=0)
                total_tok    += valid.sum().item()
                correct_tok  += (tags.view(-1)[valid]==pred.view(-1)[valid]).sum().item()
                for i in range(tags.shape[0]):
                    l = mask[i].sum().item()
                    total_morph   += 1
                    correct_morph += (tags[i,:l].tolist()==pred[i,:l].tolist())

        tok_acc   = correct_tok/max(1,total_tok)
        morph_acc = correct_morph/max(1,total_morph)
        avg_loss  = total_loss/max(1,n)

        if epoch <= 2:
            warmup_sched.step()
        else:
            plateau_sched.step(morph_acc)

        # Сохраняем лучшее состояние
        if morph_acc > best_morph:
            best_morph = morph_acc
            best_state = copy.deepcopy(model.state_dict())

        history.append({'epoch':epoch,'loss':avg_loss,
                        'tok_acc':tok_acc,'morph_acc':morph_acc})
        cur_lr = opt.param_groups[0]['lr']
        print(f'  [ep{epoch:02d}] loss={avg_loss:.4f}  '
              f'token_acc={tok_acc:.4f}  morph_acc={morph_acc:.4f}  '
              f'lr={cur_lr:.2e}')

    # Восстанавливаем лучшее состояние
    if best_state:
        model.load_state_dict(best_state)
    best = max(history, key=lambda x: x['morph_acc'])
    print(f'  Лучший результат: ep={best["epoch"]}  '
          f'morph_acc={best["morph_acc"]:.4f}')
    return history, best


# =============================================================
# ЗАПУСК ABLATION
# =============================================================

# Индивидуальный lr для каждого варианта:
# A и B быстрее сходятся — стандартный lr
# C имеет больше параметров — чуть меньший lr для стабильности
ABLATION_EPOCHS = 3   # 3 эпохи оптимально для 180–200k корпуса
dev = str(device)

configs = [
    ('A: базовый (только char)',
     dict(use_harmony=False, use_vowel=False, use_word_harm=False),
     1e-3),
    ('B: + char_harmony',
     dict(use_harmony=True, use_vowel=False, use_word_harm=False),
     1e-3),
    ('C: полный (наш)',
     dict(use_harmony=True, use_vowel=True, use_word_harm=True),
     1e-3),   # одинаковый lr для честного сравнения
]

ablation_results = []

for name, kwargs, lr in configs:
    mdl = FEMSegVariant(
        vocab_size=len(char2id),
        tagset_size=len(TAGS),
        **kwargs
    )
    hist, best = train_variant(
        name, mdl,
        train_loader, val_loader,
        num_epochs=ABLATION_EPOCHS,
        lr=lr,
        device=dev
    )
    ablation_results.append({
        'name':    name,
        'in_dim':  mdl.in_dim,
        'lr':      lr,
        'history': hist,
        'best':    best,
    })

    tag = name.split(':')[0].strip().replace(' ','_')
    abl_path = os.path.join(MODEL_DIR, f'ablation_{tag}.pt')
    torch.save({'state_dict':mdl.state_dict(),
                'config':kwargs,'best':best,'lr':lr}, abl_path)
    print(f'  Сохранён: {abl_path}')


# =============================================================
# ИТОГОВАЯ ТАБЛИЦА
# =============================================================

print()
print('='*72)
print('  ABLATION STUDY — ИТОГОВАЯ ТАБЛИЦА (лучшая эпоха каждого варианта)')
print('='*72)
print(f'  {"Вариант":<28} {"Вход":>5} {"Эп":>3} '
      f'{"token_acc":>10} {"Δ tok":>7} {"morph_acc":>10} {"Δ morph":>8}')
print('  '+'-'*70)

base_tok = base_morph = None
for r in ablation_results:
    b     = r['best']
    tok   = b['tok_acc']
    morph = b['morph_acc']
    ep    = b['epoch']
    dim   = r['in_dim']
    if base_tok is None:
        base_tok, base_morph = tok, morph
        d_tok = d_morph = '—'
    else:
        sign_t = '+' if tok   > base_tok   else ''
        sign_m = '+' if morph > base_morph else ''
        d_tok   = f'{sign_t}{(tok-base_tok)*100:.2f}%'
        d_morph = f'{sign_m}{(morph-base_morph)*100:.2f}%'
    print(f'  {r["name"]:<28} {dim:>5}d {ep:>3} '
          f'{tok:.4f} {d_tok:>7}  {morph:.4f} {d_morph:>8}')

print('='*72)
res_a = ablation_results[0]['best']
res_c = ablation_results[-1]['best']
delta_tok   = (res_c['tok_acc']   - res_a['tok_acc'])   * 100
delta_morph = (res_c['morph_acc'] - res_a['morph_acc']) * 100
sign_t = '+' if delta_tok   >= 0 else ''
sign_m = '+' if delta_morph >= 0 else ''
print(f'\n  Прирост A→C (абсолютный):')
print(f'    token_acc:  {res_a["tok_acc"]:.4f} → {res_c["tok_acc"]:.4f}  '
      f'({sign_t}{delta_tok:.2f}%)')
print(f'    morph_acc:  {res_a["morph_acc"]:.4f} → {res_c["morph_acc"]:.4f}  '
      f'({sign_m}{delta_morph:.2f}%)')

# Сохраняем результаты
abl_json = os.path.join(MODEL_DIR, 'ablation_results.json')
with open(abl_json, 'w', encoding='utf-8') as f:
    json.dump([
        {'name':r['name'],'in_dim':r['in_dim'],'lr':r['lr'],'best':r['best']}
        for r in ablation_results
    ], f, ensure_ascii=False, indent=2)
print(f'\n  Результаты: {abl_json}')


## 📈 Ячейка 12 — Визуализация кривых обучения (ablation)

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.dpi'] = 120

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = ['#888780', '#378ADD', '#1D9E75']  # серый, синий, зелёный
    labels = [r['name'] for r in ablation_results]

    for ax, metric, title in [
        (axes[0], 'tok_acc',   'Token Accuracy'),
        (axes[1], 'morph_acc', 'Morpheme Accuracy'),
    ]:
        for r, color in zip(ablation_results, colors):
            epochs = [h['epoch'] for h in r['history']]
            values = [h[metric]  for h in r['history']]
            ax.plot(epochs, values, marker='o', color=color,
                    linewidth=2, label=r['name'])
        ax.set_title(title, fontsize=13)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=max(0, min(
            h[metric] for r in ablation_results for h in r['history']
        ) - 0.005))

    plt.suptitle('FEMSeg Ablation Study — Туркменский язык, 80k корпус',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    plot_path = os.path.join(MODEL_DIR, 'ablation_curves.png')
    plt.savefig(plot_path, bbox_inches='tight')
    plt.show()
    print(f'График сохранён: {plot_path}')
except Exception as e:
    print(f'matplotlib недоступен: {e}')
    print('Запустите: !pip install matplotlib')


## 📄 Ячейка 13 — LaTeX-таблица для статьи

In [ ]:
# Генерируем готовую таблицу в формате LaTeX
lines = [
    r'\begin{table}[h]',
    r'\centering',
    r'\caption{Ablation study results for FEMSeg\_turkm on the 80k Turkmen corpus.}',
    r'\label{tab:ablation}',
    r'\begin{tabular}{lccccc}',
    r'\hline',
    r'Model Variant & Input dim & Token Acc & $\Delta$ & Morph Acc & $\Delta$ \\',
    r'\hline',
]

for idx, r in enumerate(ablation_results):
    b   = r['best']
    dim = r['in_dim']
    tok   = b['tok_acc']
    morph = b['morph_acc']
    name_short = r['name'].split('—')[0].strip() if '—' in r['name'] else r['name']
    name_short = name_short.replace('Вариант ', '').replace(':', '').strip()

    if idx == 0:
        d_tok = d_morph = '—'
    else:
        base_tok   = ablation_results[0]['best']['tok_acc']
        base_morph = ablation_results[0]['best']['morph_acc']
        sign_tok   = '+' if tok > base_tok else ''
        sign_morph = '+' if morph > base_morph else ''
        d_tok   = f'{sign_tok}{(tok-base_tok)*100:.2f}\\%'
        d_morph = f'{sign_morph}{(morph-base_morph)*100:.2f}\\%'

    # Выделяем лучшую строку (C)
    prefix = r'\textbf{' if idx == 2 else ''
    suffix = '}' if idx == 2 else ''

    lines.append(
        f'{prefix}{name_short}{suffix} & '
        f'{prefix}{dim}d{suffix} & '
        f'{prefix}{tok:.4f}{suffix} & '
        f'{prefix}{d_tok}{suffix} & '
        f'{prefix}{morph:.4f}{suffix} & '
        f'{prefix}{d_morph}{suffix} \\\\'
    )

lines += [
    r'\hline',
    r'\end{tabular}',
    r'\end{table}',
]

latex_str = '\n'.join(lines)
print(latex_str)

# Сохраняем
latex_path = os.path.join(MODEL_DIR, 'ablation_table.tex')
with open(latex_path, 'w', encoding='utf-8') as f:
    f.write(latex_str)
print(f'\nLaTeX сохранён: {latex_path}')


## 🔬 Ячейка 10 — Инференс и диагностика сингармонизма

In [ ]:
import re

# =============================================================
# NO_SPLIT — стемы которые нельзя дробить
# Включает префиксную проверку: türkmeniň → türkmen + iň
# =============================================================
NO_SPLIT_STEMS = {
    'türkmen', 'türkmenistan',
    'döwlet',  'garaşsyz',
    'mekdep',  'taýýar',
    'ýaýran',  'aýran',
}

def check_no_split(word):
    """Возвращает (hit, orig_stem, suffix) с сохранением оригинального регистра."""
    w_lower = word.lower()
    # Точное совпадение
    if w_lower in NO_SPLIT_STEMS:
        return True, word, ''
    # Префиксное совпадение
    for stem_l in sorted(NO_SPLIT_STEMS, key=len, reverse=True):
        if w_lower.startswith(stem_l) and len(word) > len(stem_l):
            return True, word[:len(stem_l)], word[len(stem_l):]
    return False, word, ''


# =============================================================
# ПОСТОБРАБОТКА МОРФЕМ v3
# =============================================================
# Аффиксы которые НЕ сливаются с соседями (легитимные короткие)
VALID_SHORT_AFFIXES = {
    'da','de','dan','den','a','e','ny','ni','dy','di',
    'lar','ler','yň','iň','sy','si','ly','li','na','ne',
}
# Неполные аффиксы — правая часть составного суффикса (слить со СЛЕДУЮЩЕЙ)
# in+de=inde, un+da=unda, ün+de=ünde, yn+da=ynda
MERGE_RIGHT_TRIGGERS = {'in', 'un', 'ün', 'yn'}

# Составные суффиксы — разбиваем на части
# lary → lar + y (мн.ч. + притяж.)
# Исправляет PARTIAL ошибки: agaçlary→agaç@@lary вместо agaç@@lar@@y
SPLIT_SUFFIXES = {
    'lary':     ['lar', 'y'],
    'lery':     ['ler', 'i'],
    'lara':     ['lar', 'a'],
    'lere':     ['ler', 'e'],
    'laryñ':    ['lar', 'yñ'],
    'leriñ':    ['ler', 'iñ'],
    'laryna':   ['lar', 'y', 'na'],
    'leryne':   ['ler', 'i', 'ne'],
    'larynyñ':  ['lar', 'y', 'nyñ'],
    'leriniñ':  ['ler', 'i', 'niñ'],
}

def postprocess_morphs(morphs):
    """
    Четыре правила:
    0. Составные суффиксы (lary/lery/...) → разбить на части
       agaç + lary  →  agaç + lar + y
    1. Неполный аффикс (in/un/ün/yn) → слить с ПРАВЫМ соседом
       köçe + ler + in + de  →  köçe + ler + inde
    2. Одиночная ñ/ň → слить с ЛЕВЫМ соседом
       stolu + ñ  →  stoluñ
    3. Одиночный символ не из VALID_SHORT → слить с ЛЕВЫМ соседом
    """
    if len(morphs) <= 1:
        return morphs

    # Проход 0: разбиваем составные суффиксы (lary → lar+y)
    expanded = []
    for m in morphs:
        m_low = m.lower()
        if m_low in SPLIT_SUFFIXES:
            expanded.extend(SPLIT_SUFFIXES[m_low])
        else:
            expanded.append(m)
    morphs = expanded

    # Проход 1: слияние вправо
    merged, i = [], 0
    while i < len(morphs):
        m = morphs[i]
        if m.lower() in MERGE_RIGHT_TRIGGERS and i + 1 < len(morphs):
            merged.append(m + morphs[i+1]); i += 2
        else:
            merged.append(m); i += 1

    # Проход 2: слияние влево
    result = [merged[0]]
    for m in merged[1:]:
        if m in ('ñ','ň','Ñ','Ň'):
            result[-1] += m
        elif len(m) == 1 and m.lower() not in VALID_SHORT_AFFIXES:
            result[-1] += m
        else:
            result.append(m)
    return result


# =============================================================
# ЗАГРУЗКА МОДЕЛИ
# =============================================================
def load_best_model(path, device='cpu'):
    ckpt = torch.load(path, map_location=device)
    c2id = ckpt['char2id']
    mdl  = TurkmFEMSegCRF(vocab_size=len(c2id), tagset_size=len(TAGS)).to(device)
    mdl.load_state_dict(ckpt['state_dict'])
    mdl.eval()
    print(f'✅ Модель: corpus={ckpt.get("corpus","?")}  '
          f'epoch={ckpt["epoch"]}  '
          f'token_acc={ckpt["val_tok"]:.4f}  '
          f'morph_acc={ckpt["val_morph"]:.4f}')
    return mdl, c2id


# =============================================================
# СЕГМЕНТАЦИЯ СЛОВА
# =============================================================
def _raw_segment(word, mdl, c2id, dev):
    chars = list(word)
    if not chars: return [word]
    def t(lst, dt): return torch.tensor([lst], dtype=dt).to(dev)
    inp  = t([c2id.get(ch, c2id['<unk>']) for ch in chars], torch.long)
    harm = t([char_harmony_class(ch) for ch in chars],       torch.long)
    vow  = t([is_vowel(ch) for ch in chars],                 torch.long)
    wh   = t([word_harmony_class(word)] * len(chars),        torch.long)
    mask = torch.ones(1, len(chars), dtype=torch.bool).to(dev)
    with torch.no_grad():
        paths = mdl(inp, harm, vow, wh, tags=None, mask=mask)
    pred_tags = [ID2TAG[tt] for tt in paths[0]]
    morphs, buf = [], []
    for ch, tag in zip(chars, pred_tags):
        buf.append(ch)
        if tag in ('E','S'): morphs.append(''.join(buf)); buf=[]
    if buf: morphs.append(''.join(buf))
    return morphs


def segment_word(word, mdl, c2id, dev='cpu'):
    """Сегментирует слово: NO_SPLIT → модель → постобработка."""
    w = norm_turkm(word)  # нормализация ň→ñ

    # NO_SPLIT с префиксной проверкой
    hit, stem, suffix = check_no_split(w)
    if hit:
        if not suffix:
            return stem
        # Сегментируем только аффикс через модель
        affix_morphs = _raw_segment(suffix, mdl, c2id, dev)
        affix_morphs = postprocess_morphs(affix_morphs)
        return '@@ '.join([stem] + affix_morphs)

    # Обычная сегментация
    morphs = _raw_segment(w, mdl, c2id, dev)
    morphs = postprocess_morphs(morphs)
    return '@@ '.join(morphs) if len(morphs) > 1 else morphs[0]


def segment_text(text, mdl, c2id, dev='cpu'):
    """Сегментирует строку. ВАЖНО: norm_turkm применяется ДО токенизации."""
    text_n = norm_turkm(text)   # <-- ключевое исправление!
    pat = re.compile(r'[a-zA-ZçñöşüýžÇÑÖŞÜÝŽ]+|[^\w\s]|\S+')
    res = []
    for tok in pat.findall(text_n):
        if re.match(r'^[a-zA-ZçñöşüýžÇÑÖŞÜÝŽ]+$', tok, re.I):
            res.append(segment_word(tok, mdl, c2id, dev))
        else:
            res.append(tok)
    return ' '.join(res)


def diagnose_harmony(mdl, c2id, test_words, dev='cpu'):
    print('\n── Диагностика сингармонизма ──────────────────────────────')
    print(f'  {"Слово":<22} {"Сегментация":<30} {"Осн.":<7} Статус')
    print('  ' + '-'*65)
    ok_count, total = 0, 0
    for word in test_words:
        seg   = segment_word(word, mdl, c2id, dev)
        parts = seg.split('@@ ')
        stem  = parts[0]
        affix = parts[1] if len(parts) > 1 else ''
        h     = 'front' if word_harmony_class(stem) == HARMONY_FRONT else 'back'
        ok    = check_harmony(stem, affix) if affix else True
        ok_count += ok; total += 1
        st = '✅' if ok else '⚠️'
        print(f'  {word:<22} -> {seg:<30} {h:<7} {st}')
    print(f'\n  Гармония корректна: {ok_count}/{total}')


# =============================================================
# ТЕСТ
# =============================================================
corpus_name = os.path.basename(CORPUS_DIR)
best_path   = os.path.join(MODEL_DIR, f'femseg_turkm_{corpus_name}_BEST.pt')

if os.path.exists(best_path):
    dev = str(device)
    best_mdl, c2id_inf = load_best_model(best_path, device=dev)

    TEST_WORDS = [
        'kitaplar',        # kitap+lar           ✓
        'adamlaryň',       # adam+lar+yň          ✓
        'ýurtda',          # ýurt+da              ✓
        'öýler',           # öý+ler               ✓
        'işçileriň',       # işçi+ler+iň          ✓
        'döwletde',        # döwlet+de            ✓
        'geldim',          # gel+dim              ✓
        'Türkmenistanda',  # Türkmenistan+da      ✓
        'köçelerinde',     # köçe+ler+inde        ✓ (было in@@de)
        'stoluň',          # stol+uň              ✓ (было stolu ñ)
        'türkmen',         # NO_SPLIT             ✓ (нет дробления)
        'türkmeniň',       # türkmen+iň           ✓ (было türkme@@niñ)
        'mekdeplerde',     # mekdep+ler+de        ✓
    ]
    diagnose_harmony(best_mdl, c2id_inf, TEST_WORDS, dev=dev)

    print('\n── Сегментация примеров ────────────────────────────────')
    texts = [
        'Türkmenistanda döwlet dili türkmen dilidir .',
        'adamlar işe gitdiler we köçelerde ýörediler .',
        'kitaplar stoluň üstünde dur .',
        'türkmeniň medeniýeti baý we dürli-dürli .',
    ]
    for txt in texts:
        print(f'  Вход:  {txt}')
        print(f'  Выход: {segment_text(txt, best_mdl, c2id_inf, dev)}')
        print()
else:
    print(f'⚠️  Модель не найдена: {best_path}')
    print('   Сначала запустите ячейку обучения.')
